In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
from ai_module.config import _embedder_config_st
import requests
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field

# Create custom storage with specific embedder
space_knowledge_storage = KnowledgeStorage(
    embedder = _embedder_config_st,
    collection_name="L014_knowledge_storage"
)

class SpaceNewsKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Space News API."""

    api_endpoint: str = Field(description="API endpoint URL")
    limit: int = Field(default=10, description="Number of articles to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format space news articles."""
        try:
            #
            response = requests.get(f"{self.api_endpoint}?limit={self.limit}")
            response.raise_for_status()

            data = response.json()
            articles = data.get('results', [])

            formatted_data = self.validate_content(articles)
            return {self.api_endpoint: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, articles: list) -> str:
        """Format articles into readable text."""
        formatted = "Space News Articles:\n\n"
        for article in articles:
            formatted += f"""
                Title: {article['title']}
                Published: {article['published_at']}
                Summary: {article['summary']}
                News Site: {article['news_site']}
                URL: {article['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)

        self._save_documents()

    def aadd(self) -> None:
        return super().aadd()    
    

# Create knowledge source
recent_news = SpaceNewsKnowledgeSource(
    api_endpoint="https://api.spaceflightnewsapi.net/v4/articles",
    limit=20,
    storage=space_knowledge_storage
)
#recent_news.storage = space_knowledge_storage


# Create specialized agent
space_analyst = Agent(
    role="Space News Analyst",
    goal="Answer questions about space news accurately and comprehensively",
    backstory="""You are a space industry analyst with expertise in space exploration,
    satellite technology, and space industry trends. You excel at answering questions
    about space news and providing detailed, accurate information.""",
    knowledge_sources=[recent_news]
)

# Create task that handles user questions
analysis_task = Task(
    description="Answer this question about space news: {user_question}",
    expected_output="A detailed answer based on the recent space news articles",
    agent=space_analyst
)

# Create and run the crew
crew = Crew(
    agents = [space_analyst],
    tasks = [analysis_task],
    verbose = True,
    process = Process.sequential,
    embedder = _embedder_config_st
)

# Example usage
result = crew.kickoff(
    inputs={"user_question": "What are the latest developments in space exploration?"}
)

/home/brijeshdhaker/IdeaProjects/bd-crewai-module/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 42e42322-e69b-4617-8bd6-96d7db922e0e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question about space news: What are the latest developments in space exploration?            │
│  ID: 50b35fbe-551d-4cfd-9b57-1a15b4f8aa7d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Space News Analyst                                                                                      │
│                                                                                                                 │
│  Task: Answer this question about space news: What are the latest developments in space exploration?            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Space News Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the recent news cycle, late May 2026 shows a multifaceted approach to space exploration,              │
│  encompassing major advancements in lunar energy storage, routine cargo resupply, deep space maneuvering, and   │
│  significant policy developments. The momentum is visible across governmental agencies, commercial providers,   │
│  and international partners.                                                                                    │
│                                                                                                                 │
│  Here is a detailed analysis of the latest developments:                                                        │
│                                                                                                                 │
│  ### **Lunar and Deep Space Exploration Campaigns**                                                             │
│                                                                                                                 │
│  The focus on the Moon and beyond remains high-priority:                                                        │
│                                                                                                                 │
│  *   **Powering the Lunar Future:** Significant steps are being taken to ensure sustainable operation on the    │
│  Moon. NASA is advancing its understanding of energy requirements with testing of **regenerative fuel cells**,  │
│  which are crucial technologies for establishing reliable energy storage at the lunar surface.                  │
│  *   **The Artemis Accords:** International collaboration continues to build, highlighted by NASA welcoming     │
│  **Paraguay** as the 67th signatory to the Artemis Accords. This expands the global commitment to shared        │
│  principles guiding civil space exploration.                                                                    │
│  *   **Mission Updates:** Other deep space initiatives are progressing. NASA’s **Psyche mission** successfully  │
│  captured an image of Mars while executing a crucial gravity assist approach, demonstrating the spacecraft's    │
│  ability to adjust its trajectory toward the target asteroid.                                                   │
│                                                                                                                 │
│  ### **Orbital Operations and Science Missions**                                                                │
│                                                                                                                 │
│  Routine operational spaceflight continues to support research and scientific data gathering:                   │
│                                                                                                                 │
│  *   **ISS Logistics:** The operational cadence is maintained via commercial resupply missions. NASA and        │
│  SpaceX are targeting the **34th Commercial Resupply Mission** with a mid-May launch aboard the Falcon 9        │
│  rocket, carrying approximately 6,500 pounds of scientific investigations, supplies, and equipment to the       │
│  International Space Station.                                                                                   │
│  *   **Scientific Imagery:** Missions are yielding incr

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question about space news: What are the latest developments in space exploration?            │
│  Agent: Space News Analyst                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ e44257eb-3fac-4946-ac65-999e6e7405d4                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/e44257eb-3fac-494 │
│ 6-ac65-999e6e7405d4?access_code=TRACE-18701d5c48                             │
│ 🔑 Access Code: TRACE-18701d5c48                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 42e42322-e69b-4617-8bd6-96d7db922e0e                                                                       │
│  Final Output: Based on the recent news cycle, late May 2026 shows a multifaceted approach to space             │
│  exploration, encompassing major advancements in lunar energy storage, routine cargo resupply, deep space       │
│  maneuvering, and significant policy developments. The momentum is visible across governmental agencies,        │
│  commercial providers, and international partners.                                                              │
│                                                                                                                 │
│  Here is a detailed analysis of the latest developments:                                                        │
│                                                                                                                 │
│  ### **Lunar and Deep Space Exploration Campaigns**                                                             │
│                                                                                                                 │
│  The focus on the Moon and beyond remains high-priority:                                                        │
│                                                                                                                 │
│  *   **Powering the Lunar Future:** Significant steps are being taken to ensure sustainable operation on the    │
│  Moon. NASA is advancing its understanding of energy requirements with testing of **regenerative fuel cells**,  │
│  which are crucial technologies for establishing reliable energy storage at the lunar surface.                  │
│  *   **The Artemis Accords:** International collaboration continues to build, highlighted by NASA welcoming     │
│  **Paraguay** as the 67th signatory to the Artemis Accords. This expands the global commitment to shared        │
│  principles guiding civil space exploration.                                                                    │
│  *   **Mission Updates:** Other deep space initiatives are progressing. NASA’s **Psyche mission** successfully  │
│  captured an image of Mars while executing a crucial gravity assist approach, demonstrating the spacecraft's    │
│  ability to adjust its trajectory toward the target asteroid.                                                   │
│                                                                                                                 │
│  ### **Orbital Operations and Science Missions**                                                                │
│                                                                                                                 │
│  Routine operational spaceflight continues to support research and scientific data gathering:                   │
│                                                                                                                 │
│  *   **ISS Logistics:** The operational cadence is maintained via commercial resupply missions. NASA and        │
│  SpaceX are targeting the **34th Commercial Resupply Mission** with a mid-May launch aboard the Falcon 9        │
│  rocket, carrying approximately 6,500 pounds of scientific investigations, supplies, and equipment to the       │
│  International Space Station.                                                                                   │
│  *   **Scientific Imagery:** Missions are yielding inc